In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)


In [ ]:
# Reading the saved data pickle file
rng = pd.date_range('2019-02-24', periods=661, freq='T')
df  = pd.DataFrame()
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research//master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research//master/financialmodelingprep/data/pkl/sentiment.pkl', compression=None)
df_companies=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/press_release_companies_statistics.pkl', compression=None)

In [ ]:
df["articles"].describe()
df = df.join(df_sentiment)
df[["VIX"]] = df["adj close"].apply(np.int)
df["log_VIX"]=np.log(df["VIX"]/df["VIX"].shift(1))
df["sentiment_ewm"]=df["sentiment"].ewm(20).mean()

In [ ]:
df

In [ ]:
df["sentiment"]

In [ ]:
# 2. Foreign volatility indices
corr = df.loc[:,['VIX', 'sentiment_ewm']].corr()

# heatmap
mask = np.zeros_like(corr)
for i in range(mask.shape[0]):
    mask[i,i]=True
sns.heatmap(corr,annot=True,mask=mask,cmap='coolwarm',vmin=-1,vmax=1)
plt.show()

In [ ]:
sentence = 'The shooting in Paris killed two incon'
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
sid
ss = sid.polarity_scores(sentence)
ss

In [ ]:
df.loc['2020-06-01':'2021-07-16']

In [ ]:
for date, sentiment in df.T.iteritems():
    print(date)
    # if (df[date, sentiment]==''):
    #     print("fdsafsda")

In [ ]:
#df.diff().hist(color="k", alpha=0.5, bins=50);
names = ['negative','positive']
values = [np.sum(df['neg']), np.sum(df['pos'])]

plt.figure(figsize=(9, 3))

#plt.subplot(131)
plt.bar(names, values)

In [ ]:
# function to add value labels
# function to add value labels
def addlabels(x,y):
    for i in range(len(x)):
        plt.text(i,1,y[i])

names = ['negative','neutral', 'positive']
values = [np.sum(df['neg']), np.sum(df['neu']), np.sum(df['pos'])]
#addlabels(names, values)
#plt.text(1, values[1], values[1], ha = 'center')
plt.figure()#figsize=(8, 3),
plt.xlabel("Sentiment Type")
plt.ylabel("Percentage of words")
plt.bar(names, values)

In [ ]:
# Creating plot
data=df[['sentiment', 'sentiment_ewm']]
plt.boxplot(data)

plt.xticks([1, 2], ['sentiment', 'sentiment_ewm'])
#plt.bar(names, values)
# # show plot
plt.show()

In [ ]:
# Creating plot
data=df[['VIX']]
plt.boxplot(data)

plt.xticks([1], ['VIX'])
#plt.bar(names, values)
# # show plot
plt.show()

In [ ]:
df["sentiment"].dropna(how='all', inplace = True)#values#.to_string().(" ")
#df[['date' 'sentiment']] = pd.DataFrame([ x.split(' ') for x in df['_id'].tolist() ])


In [ ]:
df[['sentiment']]

In [ ]:
df_part3 = df.loc['2020-06-01':'2021-07-16']

In [ ]:
# import plotly.express as px

# df_part3[['sentiment']] =df_part3[['sentiment']].apply(pd.to_numeric)
# df_part3[['sentiment']].plot()
# dfplot = df[['sentiment']].plot(title='sentiment by date')
# fig = dfplot.get_figure()
# fig.update_xaxes(
#     dtick="M1",
#     tickformat="%b\n%Y",
#     ticklabelmode="period")

# fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentiment.png")
# df_part3[['sentiment']].ewm(span=20).mean().plot()
# fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentiment_ewm.png")



In [ ]:
df[['sentiment_complementary']] =df[['sentiment_complementary']].apply(pd.to_numeric)
df[['sentiment_complementary']].plot()
fig, ax = plt.subplots()

fig.savefig("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/images/sentiment_reflection.png")
df[['sentiment_complementary']].ewm(span=20).mean().plot()
#ts=pd.Series(df['sentiment'], index=pd.date_range("25/09/2019", periods=569))
#ts.plot()
# dfplot = df[['sentiment']].plot(title='sentiment by date').apply(pd.to_numeric) 
# fig = dfplot.get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/images/sentiment_reflection_ewm.png")

In [ ]:
import matplotlib.pyplot as plt
#df[['sentiment']] = df[['sentiment']].ewm(span=20)
fig, ax1 = plt.subplots(figsize=(20,15))
ax1.plot(df[['VIX']])
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("Sentiment and VIX", size=26)
ax1.set_ylabel("VIX", size=20)
ax2 = ax1.twinx()
ax2.plot(df[['sentiment']].ewm(span=20).mean(), color='#f97306')# color='#e50000')
ax2.set_ylim([50, 100])
ax2.set_ylabel("sentiment", size=20)
fig.autofmt_xdate()
fig = ax2.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/images/sentimentVIX.png")

In [ ]:
import matplotlib.pyplot as plt
#df[['sentiment']] = df[['sentiment']].ewm(span=20)
fig, ax1 = plt.subplots(figsize=(20,15))
ax1.plot(df[['VIX']], color='#f97306')
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("Sentiment Complementary and VIX", size=26)
ax1.set_ylabel("VIX", size=20)
ax3 = ax1.twinx()
ax3.plot(df[['sentiment_complementary']].ewm(span=20).mean())# color='#e50000')
ax3.set_ylim([0, 50])
ax3.set_ylabel("sentiment_complementary", size=20)
fig.autofmt_xdate()
fig = ax3.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentimentComplementaryVIX.png")

In [ ]:
import pandas as pd
#if you get an error after executing the code, try adding below:
pd.core.common.is_list_like = pd.api.types.is_list_like

import pandas_datareader.data as web
import datetime

start = datetime.datetime(2019, 9, 25)
end = datetime.datetime(2021, 6, 24)

SP500 = web.DataReader(['sp500'], 'fred', start, end)

In [ ]:
SP500.plot(title='S&P 500 daily returns')

In [ ]:
import matplotlib.pyplot as plt
#df[['sentiment']] = df[['sentiment']].ewm(span=20)
fig, ax1 = plt.subplots(figsize=(20,15))
ax1.plot(SP500, color='#f97306')
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("VIX and S&P500", size=26)
ax1.set_ylabel("S&P500", size=20)
ax1.set_ylim([2000, 4500])
ax3 = ax1.twinx()
ax3.plot(df[['VIX']])# color='#e50000')
ax3.set_ylim([0, 100])
ax3.set_ylabel("VIX", size=20)
fig.autofmt_xdate()
fig = ax3.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/VIXandS&P500.png")

In [ ]:
SP500['daily_return'] = np.log(SP500['sp500']/SP500['sp500'].shift(1))
df['vix_ln_return'] = np.log(df['VIX']/df['VIX'].shift(1))

#Drop all Not a number values using drop method.
SP500.dropna(inplace = True)

In [ ]:
SP500['daily_return'].ewm(span=20).mean().plot(title='S&P 500 daily returns')
df['vix_ln_return_ewm100']=df['vix_ln_return'].ewm(span=100).mean()#.plot(title='vix ln returns')

In [ ]:
fig, axs = plt.subplots(4,figsize=(20,15))
fig.suptitle('Sentiment and VIX', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("vix_ln_return_ewm100", size=20)
axs[0].plot(df[['vix_ln_return_ewm100']])
axs[1].set_ylabel("sentiment", size=20)
axs[1].plot(df[['sentiment']].ewm(span=20).mean())
axs[2].set_ylabel("S&P500 daily_return", size=20)
axs[2].plot(SP500['daily_return'].ewm(span=20).mean())
axs[3].set_ylabel("S&P500", size=20)
axs[3].plot(SP500['sp500'])
fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentimentAndVIX.png")

In [ ]:
fig, axs = plt.subplots(3,figsize=(20,15))
fig.suptitle('S&P500 and S&P500 daily return', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("S&P500", size=20)
axs[0].plot(SP500['sp500'])
# axs[1].set_ylabel("S&P500 Simple Return", size=20)
# axs[1].plot(SP500['daily_return'].ewm(span=20).mean())
SP500['sp500_log']=np.log(SP500['sp500']/SP500['sp500'].shift(1))

axs[1].set_ylabel("S&P500 Log Return", size=20)
axs[1].plot(SP500['sp500_log'])


axs[2].set_ylabel("S&P500 Log Return EWM", size=20)
axs[2].plot(SP500['sp500_log'].ewm(span=20).mean())
fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentimentAndVIX.png")

In [ ]:
fig, axs = plt.subplots(3,figsize=(20,15))
fig.suptitle('Sentiment and VIX', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("VIX", size=20)
axs[0].plot(df[['VIX']])
axs[1].set_ylabel("sentiment", size=20)
axs[1].plot(df[['sentiment']].ewm(span=20).mean())
axs[2].set_ylabel("S&P500 daily return", size=20)
axs[2].plot(SP500['daily_return'].ewm(span=20).mean())
fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentimentAndVIX.png")

In [ ]:
fig, axs = plt.subplots(2,figsize=(20,15))
fig.suptitle('Sentiment Complementary and VIX', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("VIX", size=20)
axs[0].plot(df[['VIX']])
axs[1].set_ylabel("sentiment_complementary", size=20)
axs[1].plot(df[['sentiment_complementary']].ewm(span=20).mean())
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentimentComplementaryAndVIX.png")

In [ ]:
df[['VIX', 'compound', 'neg', 'neu', 'pos', 'sentiment']]

In [ ]:
#df_sentiment = df['sentiment'].apply(np.int)

In [ ]:
#df_sentiment

In [ ]:
sentence = 'The shooting in Paris killed two'
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
sid
ss = sid.polarity_scores(sentence)
ss

In [ ]:
# import plotly.express as px
# fig = px.box(df, y="VIX")
# fig.show()
df
#sns.boxplot(y='neu', data=df)




# df_sentiment['sentiment'] = df
# target_dist = sns.factorplot(x='data',data=df_sentiment, kind="count", size=20, aspect=1.5, palette="PuBuGn_d")
#target_dist.plot()
# plt.show();
# target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/target_dist_vix.png')
# target_dist = sns.factorplot(x="sentiment", data=df, kind="count", size=20, aspect=1.5, palette="PuBuGn_d")
# plt.show();
# target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/target_dist_sentiment.png')

In [ ]:
# mu, sigma = 100, 15
# #x = mu + sigma * df['VIX']

# # the histogram of the data
# n, bins, patches = plt.hist(df['VIX'])


# plt.xlabel('Smarts')
# plt.ylabel('Probability')
# plt.title('Histogram of IQ')
# plt.text(100, .025, r'$\mu=100,\ \sigma=15$')
# plt.axis([0, 100, 0, 200])
# plt.grid(True)
# plt.show()

In [ ]:
# fig, axes = plt.subplots(10,2,figsize=(25,75))
# # for i,col in enumerate(df['VIX']):
# sns.regplot(1,'VIX',data=df, ax=axes[i,0]) # regression plot
# sns.kdeplot(df['VIX'],df['sentiment'],shade=True, ax=df[i,1]) # kernel density e

In [ ]:
# # Plot each market variable with VKOSPI

# # Columns to plot 
# plot_columns = ['sentiment','VIX']

# # plot
# fig, axes = plt.subplots(10,2,figsize=(25,75))
# for i,col in enumerate(plot_columns):
#     sns.regplot(col,'VIX',data=df, ax=axes[i,0]) # regression plot
#     sns.kdeplot(df[col],df[['VIX']],shade=True, ax=axes[i,1]) # kernel density estimate plot

In [ ]:
# def cor_reg_kde(x,y,data):
#     # Correlation coefficient
#     #print('correlation coefficient:',corr.loc[x,y])
    
#     # Subplot adjust
#     plt.subplots_adjust(wspace=0.5)
    
#     # Plot (regression and kernel density)
#     fig,(ax1,ax2) = plt.subplots(1,2,figsize=(10,5))
#     sns.regplot(x,y,data=data,ax=ax1)
#     sns.kdeplot(data[[x]],data[y],shade=True,ax=ax2)
    
# cor_reg_kde('pos','VIX',df)
# # cor_reg_kde('Indiv_Put_Netbuying_Quantity','Open_interest',df)
# # cor_reg_kde('Indiv_Put_Netbuying_Quantity','Indiv_Call_Netbuying_Quantity',df)
# # cor_reg_kde('Indiv_Call_Netbuying_Quantity','Open_interest',df)

In [ ]:
# # Checking date for empty strings
# for date, row in df_vix.T.iteritems():
#     if type(df_vix.loc[date, 'articles']).__name__ == 'str':
#         print(date)

In [ ]:
# df

In [ ]:
class TextCounts(BaseEstimator, TransformerMixin):
    
    def count_regex(self, pattern, tweet):
        return len(re.findall(pattern, tweet))
    
    def fit(self, X, y=None, **fit_params):
        # fit method is used when specific operations need to be done on the train data, but not on the test data
        return self
    
    def transform(self, X, **transform_params):
        count_words = X.apply(lambda x: self.count_regex(r'\w+', x)) 
        count_mentions = X.apply(lambda x: self.count_regex(r'@\w+', x))
        count_hashtags = X.apply(lambda x: self.count_regex(r'#\w+', x))
        count_capital_words = X.apply(lambda x: self.count_regex(r'\b[A-Z]{2,}\b', x))
        count_excl_quest_marks = X.apply(lambda x: self.count_regex(r'!|\?', x))
        count_urls = X.apply(lambda x: self.count_regex(r'http.?://[^\s]+[\s]?', x))
        # Moreover, it will result in having more words in the tweet
        
        df = pd.DataFrame({'count_words': count_words
                           , 'count_mentions': count_mentions
                           , 'count_hashtags': count_hashtags
                           , 'count_capital_words': count_capital_words
                           , 'count_excl_quest_marks': count_excl_quest_marks
                           , 'count_urls': count_urls
                          })
        
        return df

In [ ]:
tc = TextCounts()
#df_eda = tc.fit_transform(df_vix.articles)


In [ ]:
def show_dist(df, col):
    print('Descriptive stats for {}'.format(col))
    print('-'*(len(col)+22))
    print(df.groupby('VIX')[col].describe())
    bins = np.arange(df[col].min(), df[col].max() + 1)
    g = sns.FacetGrid(df, col='VIX', size=5, hue='VIX', palette="PuBuGn_d")
    g = g.map(sns.distplot, col, kde=False, norm_hist=True, bins=bins)
    plt.show()

In [ ]:
#df_vixs=df[['VIX']]
df_vixs=df
target_dist = sns.factorplot(x="VIX", data=df_vixs, kind="count", size=10, aspect=1.5, palette="PuBuGn_d")
plt.show();
target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/VIX_Histogram.png')

In [ ]:
#df[['sentiment']])#.add(0.5).round().apply(int).options.display.float_format = '{:,.0f}'.format
#round()
df[['sentiment']] = pd.to_numeric(df['sentiment'])

In [ ]:
df[['sentiment']].round()

In [ ]:
target_dist = sns.factorplot(x="sentiment", data=df[['sentiment']].round(0), kind="count", size=15, aspect=1.5, palette="PuBuGn_d")
plt.show();
target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentiment_Histogram.png')

In [ ]:
#show_dist(df_vixs, 'VIX')

In [ ]:
# Time series data source: fpp pacakge in R.
import matplotlib.pyplot as plt
df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/a10.csv', parse_dates=['date'], index_col='date')

# Draw Plot
def plot_df(df, x, y, title="", xlabel='Date', ylabel='Value', dpi=100):
    plt.figure(figsize=(16,5), dpi=dpi)
    plt.plot(x, y, color='tab:red')
    plt.gca().set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.show()

plot_df(df, x=df.index, y=df.value, title='Monthly anti-diabetic drug sales in Australia from 1992 to 2008.')    

In [ ]:
# Import Data
df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/a10.csv', parse_dates=['date'], index_col='date')
df.reset_index(inplace=True)

# Prepare data
df['year'] = [d.year for d in df.date]
df['month'] = [d.strftime('%b') for d in df.date]
years = df['year'].unique()

# Draw Plot
fig, axes = plt.subplots(1, 2, figsize=(20,7), dpi= 80)
sns.boxplot(x='year', y='value', data=df, ax=axes[0])
sns.boxplot(x='month', y='value', data=df.loc[~df.year.isin([1991, 2008]), :])

# Set Title
axes[0].set_title('Year-wise Box Plot\n(The Trend)', fontsize=18); 
axes[1].set_title('Month-wise Box Plot\n(The Seasonality)', fontsize=18)
plt.show()

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,4), dpi=100)
pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/guinearice.csv', parse_dates=['date'], index_col='date').plot(title='Trend Only', legend=False, ax=axes[0])

pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/sunspotarea.csv', parse_dates=['date'], index_col='date').plot(title='Seasonality Only', legend=False, ax=axes[1])

pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/AirPassengers.csv', parse_dates=['date'], index_col='date').plot(title='Trend and Seasonality', legend=False, ax=axes[2])